# Tokenizer probe: mE5-large + morfem sınırları (TurkishTokenizer / Morpheus)

`purpose=tokenizer_probe_only`: sealed 600 bu deneyde eğitim/seçim için kullanılır, sonuçlar sealed-test sonucu değildir.
Önkoşul: yerel `experiments/tokenizer_probe/artifacts/` klasörü Drive'a `MyDrive/tokenizer_probe_artifacts` olarak kopyalanmış olmalı (repo'ya push edilmez).

## 1. Kurulum

In [ ]:
!pip uninstall -y -q torchvision torchao
!pip install -q -U sentence-transformers transformers peft accelerate datasets pandas scikit-learn matplotlib
import subprocess, sys, shutil, torch
from pathlib import Path
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
assert torch.cuda.is_available(), 'GPU runtime seçin.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git'
BRANCH = 'tokenizer-probe'
ROOT = Path('/content/turkish-morph-retrieval')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/turkish-morph-retrieval
# artifacts sealed veriden türediği için repo'da değil, Drive'da tutulur.
ARTIFACTS_DRIVE = Path('/content/drive/MyDrive/tokenizer_probe_artifacts')
assert (ARTIFACTS_DRIVE / 'seg_mph.json').exists(), f'{ARTIFACTS_DRIVE} içinde artifacts yok'
shutil.copytree(ARTIFACTS_DRIVE, 'experiments/tokenizer_probe/artifacts', dirs_exist_ok=True)
OUT = '/content/drive/MyDrive/tokenizer_probe_runs'  # devam-güvenli: biten koşular atlanır

## 2. Smoke testi (fold 0, küçük alt küme, 1 epoch, her kol)

In [ ]:
!python -m experiments.tokenizer_probe.train_eval --stage folds --arms base tt mph --smoke --out $OUT

## 3. Aşama A: zero-shot, tüm 600 aile (kontrol kolları dahil)

In [ ]:
!python -m experiments.tokenizer_probe.train_eval --stage zero_shot --arms base tt mph shift_tt shift_mph rand_tt rand_mph --out $OUT

## 4. Aşama B: base / tt / mph × 5 fold × seed 42 (15 koşu)

In [ ]:
!python -m experiments.tokenizer_probe.train_eval --stage folds --arms base tt mph --folds 0 1 2 3 4 --seeds 42 --out $OUT

## 5. Rapor

In [ ]:
!python -m experiments.tokenizer_probe.report --runs $OUT --out $OUT/report
from IPython.display import Markdown, Image, display
display(Markdown(open(OUT + '/report/report.md').read()))
display(Image(OUT + '/report/per_phenomenon_heatmap.png'))

## 6. Aşama C (yalnız karar kuralı gerektiriyorsa)
Fark varsa veya belirsizse: ek seed'ler (`--seeds 1 2`) ve kontrol kolları (`--arms shift_tt shift_mph`). Kontroller non-morfolojik sınırların zararını gösterir; daha sıkı kontrol (SentencePiece örneklemeli, eşit uzunluk artışı) gerekirse eklenir.